# MobileNetV2 Post-Training Quantization — CIFAR-10

This notebook implements and evaluates configurable uniform linear PTQ for MobileNetV2 weights and activations.

## 1. Environment and FP32 checkpoint

In [ ]:
import os
import copy
import math
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import mobilenet_v2

print("PyTorch version:", torch.__version__)
print("Using device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

For a repository-based run, the same quantization implementation is also available in `src/quantization.py`, and `scripts/evaluate.py` provides a command-line evaluation interface.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:

MODEL_PATH = "/content/drive/MyDrive/CS6886_MobileNetV2/mobilenetv2_cifar10_fp32.pth"

print("Model exists:", os.path.exists(MODEL_PATH))

if os.path.exists(MODEL_PATH):
    print("Model size:",
          round(os.path.getsize(MODEL_PATH) / (1024 * 1024), 2),
          "MB")
else:
    print("ERROR: Saved model was not found.")

## 2. Dataset and baseline evaluation

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("Test images:", len(test_dataset))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_model = mobilenet_v2(num_classes=10)

best_model.classifier[1] = nn.Linear(
    best_model.last_channel,
    10
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    best_model.load_state_dict(checkpoint["model_state_dict"])
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    best_model.load_state_dict(checkpoint["state_dict"])
else:
    best_model.load_state_dict(checkpoint)

best_model = best_model.to(device)
best_model.eval()

print("FP32 model loaded successfully.")
print("Using device:", device)

In [ ]:
def evaluate_model(model, loader, device):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return 100.0 * correct / total




## 3. Linear quantization implementation

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F


def get_qrange(bits, symmetric):
    """Return integer quantization range."""
    if bits < 2:
        raise ValueError("bits must be >= 2")

    if symmetric:
        qmin = -(2 ** (bits - 1))
        qmax = (2 ** (bits - 1)) - 1
    else:
        qmin = 0
        qmax = (2 ** bits) - 1

    return qmin, qmax


def calculate_qparams(tensor, bits=8, symmetric=True, per_channel=False):
    """
    Calculate linear quantization parameters.

    symmetric=True:
        Signed symmetric quantization, zero_point = 0.

    symmetric=False:
        Unsigned affine/asymmetric quantization.

    per_channel=True:
        One scale per output channel.
        For Conv2d/Linear weights, channel dimension = 0.
    """

    qmin, qmax = get_qrange(bits, symmetric)

    x = tensor.detach()

    if per_channel:
        # Quantize each output channel independently.
        reduce_dims = tuple(range(1, x.ndim))

        x_min = x.amin(dim=reduce_dims, keepdim=True)
        x_max = x.amax(dim=reduce_dims, keepdim=True)
    else:
        x_min = x.min()
        x_max = x.max()

    if symmetric:
        max_abs = torch.maximum(x_min.abs(), x_max.abs())

        scale = max_abs / float(qmax)

        scale = torch.where(
            scale == 0,
            torch.ones_like(scale),
            scale
        )

        zero_point = torch.zeros_like(scale)

    else:
        scale = (x_max - x_min) / float(qmax - qmin)

        scale = torch.where(
            scale == 0,
            torch.ones_like(scale),
            scale
        )

        zero_point = torch.round(
            qmin - x_min / scale
        ).clamp(qmin, qmax)

    return scale, zero_point, qmin, qmax


def fake_quantize(
    tensor,
    scale,
    zero_point,
    qmin,
    qmax
):
    """
    Quantize to integer levels and immediately dequantize.

    This is intentional fake quantization used for PTQ
    accuracy evaluation.
    """

    q = torch.round(
        tensor / scale + zero_point
    )

    q = q.clamp(qmin, qmax)

    x_hat = (
        q - zero_point
    ) * scale

    return x_hat

In [ ]:
class QuantizedConv2d(nn.Module):

    def __init__(
        self,
        original_layer,
        bits=8,
        per_channel=True
    ):
        super().__init__()

        self.layer = original_layer
        self.bits = bits
        self.per_channel = per_channel

        scale, zero_point, qmin, qmax = calculate_qparams(
            self.layer.weight,
            bits=bits,
            symmetric=True,
            per_channel=per_channel
        )

        self.register_buffer("scale", scale)
        self.register_buffer("zero_point", zero_point)

        self.qmin = qmin
        self.qmax = qmax

    def forward(self, x):

        q_weight = fake_quantize(
            self.layer.weight,
            self.scale,
            self.zero_point,
            self.qmin,
            self.qmax
        )

        return F.conv2d(
            x,
            q_weight,
            self.layer.bias,
            self.layer.stride,
            self.layer.padding,
            self.layer.dilation,
            self.layer.groups
        )


class QuantizedLinear(nn.Module):

    def __init__(
        self,
        original_layer,
        bits=8,
        per_channel=True
    ):
        super().__init__()

        self.layer = original_layer
        self.bits = bits
        self.per_channel = per_channel

        scale, zero_point, qmin, qmax = calculate_qparams(
            self.layer.weight,
            bits=bits,
            symmetric=True,
            per_channel=per_channel
        )

        self.register_buffer("scale", scale)
        self.register_buffer("zero_point", zero_point)

        self.qmin = qmin
        self.qmax = qmax

    def forward(self, x):

        q_weight = fake_quantize(
            self.layer.weight,
            self.scale,
            self.zero_point,
            self.qmin,
            self.qmax
        )

        return F.linear(
            x,
            q_weight,
            self.layer.bias
        )

In [ ]:
class ActivationFakeQuant(nn.Module):

    def __init__(self, bits=8):
        super().__init__()

        self.bits = bits

        self.calibrating = True
        self.frozen = False

        self.register_buffer("min_value", torch.tensor(float("inf")))
        self.register_buffer("max_value", torch.tensor(float("-inf")))
        self.register_buffer("scale", torch.tensor(1.0))
        self.register_buffer("zero_point", torch.tensor(0.0))

    def forward(self, x):

        if self.calibrating:

            self.min_value.copy_(
                torch.minimum(
                    self.min_value,
                    x.detach().min()
                )
            )

            self.max_value.copy_(
                torch.maximum(
                    self.max_value,
                    x.detach().max()
                )
            )

        if not self.frozen:
            return x

        qmin, qmax = get_qrange(
            self.bits,
            symmetric=False
        )

        return fake_quantize(
            x,
            self.scale,
            self.zero_point,
            qmin,
            qmax
        )

    def freeze(self):

        observed_range = torch.stack([
            self.min_value,
            self.max_value
        ])

        scale, zero_point, qmin, qmax = calculate_qparams(
            observed_range,
            bits=self.bits,
            symmetric=False,
            per_channel=False
        )

        self.scale.copy_(scale)
        self.zero_point.copy_(zero_point)

        self.calibrating = False
        self.frozen = True

In [ ]:
def replace_with_quantized_modules(
    model,
    weight_bits=8,
    activation_bits=8,
    weight_per_channel=True
):
    """Recursively replace Conv2d/Linear and ReLU/ReLU6 modules."""
    for name, child in list(model.named_children()):
        if isinstance(child, nn.Conv2d):
            setattr(
                model,
                name,
                QuantizedConv2d(
                    child,
                    bits=weight_bits,
                    per_channel=weight_per_channel
                )
            )
        elif isinstance(child, nn.Linear):
            setattr(
                model,
                name,
                QuantizedLinear(
                    child,
                    bits=weight_bits,
                    per_channel=weight_per_channel
                )
            )
        elif isinstance(child, (nn.ReLU, nn.ReLU6)):
            setattr(
                model,
                name,
                nn.Sequential(
                    child,
                    ActivationFakeQuant(bits=activation_bits)
                )
            )
        else:
            replace_with_quantized_modules(
                child,
                weight_bits=weight_bits,
                activation_bits=activation_bits,
                weight_per_channel=weight_per_channel
            )
    return model

## 4. Calibration data for activation PTQ

In [ ]:
calibration_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

calibration_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=calibration_transform
)

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("Calibration loader created successfully.")
print("Calibration images:", len(calibration_dataset))

## 5. Quantization sweep (Q3)

In [ ]:
import time
import pandas as pd

experiment_configs = [
    (8, 8, True),
    (6, 8, True),
    (4, 8, True),
    (8, 6, True),
    (8, 4, True),
    (6, 6, True),
    (4, 4, True),
]

def run_quantization_experiment(
    fp32_model, test_loader, calibration_loader, device,
    weight_bits, activation_bits, weight_per_channel=True, calibration_batches=100
):
    start_time = time.time()
    model = copy.deepcopy(fp32_model)
    model = replace_with_quantized_modules(
        model, weight_bits=weight_bits, activation_bits=activation_bits,
        weight_per_channel=weight_per_channel
    )
    model = model.to(device).eval()

    for module in model.modules():
        if isinstance(module, ActivationFakeQuant):
            module.calibrating = True
            module.frozen = False
            module.min_value.fill_(float("inf"))
            module.max_value.fill_(float("-inf"))

    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(calibration_loader):
            model(images.to(device))
            if batch_idx + 1 >= calibration_batches:
                break

    for module in model.modules():
        if isinstance(module, ActivationFakeQuant):
            module.freeze()

    accuracy = evaluate_model(model, test_loader, device)
    accuracy_drop = fp32_accuracy - accuracy
    return {
        "weight_bits": weight_bits,
        "activation_bits": activation_bits,
        "weight_granularity": "per-channel" if weight_per_channel else "per-tensor",
        "accuracy": accuracy,
        "accuracy_drop": accuracy_drop,
        "time_seconds": time.time() - start_time
    }

experiment_results = []
for w_bits, a_bits, per_channel in experiment_configs:
    result = run_quantization_experiment(
        best_model, test_loader, calibration_loader, device,
        w_bits, a_bits, per_channel, calibration_batches=100
    )
    experiment_results.append(result)
    print(f"W{w_bits}/A{a_bits}: {result['accuracy']:.2f}% | drop {result['accuracy_drop']:.2f} pp")

results_df = pd.DataFrame(experiment_results)
print("\n========== EXPERIMENT RESULTS ==========")
display(results_df)

## 6. Storage accounting

In [ ]:
def count_activation_sites(model):
    return sum(1 for module in model.modules() if isinstance(module, (nn.ReLU, nn.ReLU6)))

def calculate_realistic_storage(model, weight_bits, activation_bits, weight_per_channel=True):
    fp32_parameter_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    fp32_buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    fp32_total_bytes = fp32_parameter_bytes + fp32_buffer_bytes

    quantized_weight_bytes = 0
    quantized_weight_elements = 0
    weight_scale_bytes = 0
    for _, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            num_weights = module.weight.numel()
            quantized_weight_elements += num_weights
            quantized_weight_bytes += num_weights * weight_bits / 8.0
            num_scales = module.weight.shape[0] if weight_per_channel else 1
            weight_scale_bytes += num_scales * 4

    total_parameter_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    remaining_parameter_bytes = total_parameter_bytes - quantized_weight_elements * 4
    activation_sites = count_activation_sites(model)
    activation_metadata_bytes = activation_sites * 8
    compressed_bytes = (
        quantized_weight_bytes + weight_scale_bytes + remaining_parameter_bytes
        + fp32_buffer_bytes + activation_metadata_bytes
    )
    weight_fp32_bytes = quantized_weight_elements * 4
    return {
        "weight_bits": weight_bits,
        "activation_bits": activation_bits,
        "compressed_size_MB": compressed_bytes / (1024 ** 2),
        "weight_compression_ratio": weight_fp32_bytes / quantized_weight_bytes,
        "activation_compression_ratio": 32.0 / activation_bits,
        "overall_compression_ratio": fp32_total_bytes / compressed_bytes,
        "activation_sites": activation_sites
    }

storage_results = []
for _, row in results_df.iterrows():
    storage = calculate_realistic_storage(best_model, int(row['weight_bits']), int(row['activation_bits']), True)
    storage_results.append({
        "weight_bits": int(row['weight_bits']),
        "activation_bits": int(row['activation_bits']),
        "weight_granularity": row['weight_granularity'],
        "accuracy": row['accuracy'],
        "accuracy_drop": row['accuracy_drop'],
        **storage,
    })

full_results_df = pd.DataFrame(storage_results)
print("========== COMPLETE Q3 RESULTS ==========")
display(full_results_df.round(4))

## 7. Activation footprint measurement

In [ ]:
def measure_activation_footprint(fp32_model, test_loader, device, activation_bits):
    model = copy.deepcopy(fp32_model)
    model = replace_with_quantized_modules(model, weight_bits=8, activation_bits=activation_bits, weight_per_channel=True)
    model = model.to(device).eval()
    captured_elements = []
    hooks = []
    def capture_activation(module, inputs, output):
        captured_elements.append(output.numel())
    for module in model.modules():
        if isinstance(module, ActivationFakeQuant):
            hooks.append(module.register_forward_hook(capture_activation))
    images, labels = next(iter(test_loader))
    images = images.to(device)
    with torch.no_grad():
        model(images)
    for hook in hooks:
        hook.remove()
    total_elements = sum(captured_elements)
    fp32_bytes = total_elements * 4
    quantized_bytes = total_elements * activation_bits / 8.0
    return {
        "activation_bits": activation_bits,
        "activation_sites": len(captured_elements),
        "total_activation_elements": total_elements,
        "FP32_activation_MB": fp32_bytes / (1024 ** 2),
        "quantized_activation_MB": quantized_bytes / (1024 ** 2),
        "activation_compression_ratio": fp32_bytes / quantized_bytes
    }

activation_measurements = [measure_activation_footprint(best_model, test_loader, device, bits) for bits in [8, 6, 4]]
activation_df = pd.DataFrame(activation_measurements)
print("========== ACTIVATION MEASUREMENTS ==========")
display(activation_df.round(4))

## 8. Final operating point (Q4)

In [ ]:
w6a8_storage = calculate_realistic_storage(best_model, 6, 8, True)
print("\n========== FINAL Q4: W6/A8 ==========")
for key, value in w6a8_storage.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")